#  Multi-Model Benchmarking & Evaluation Matrix — Google Colab Edition

Evaluate all 5 Stable Diffusion model configurations on 5 performance and quality metrics (CLIP, FID, Speed, VRAM, and Edge Density) side-by-side on a live prompt.

 **Runtime Requirement**: Select **T4 GPU** (or any GPU) to execute.


### Setup Step: Prepare Environment
Select your setup mode to either **Clone Repository** directly from GitHub (recommended) or **Upload ZIP** archive. 
If you are running modular pipeline notebooks, enable **UPLOAD_PREVIOUS_OUTPUTS** to upload your outputs from previous stages.

In [ ]:
#@title Choose Setup Method { run: "auto" }
SETUP_MODE = "git" #@param ["git", "zip"]
UPLOAD_PREVIOUS_OUTPUTS = True #@param {type:"boolean"}

import os, subprocess, zipfile
from google.colab import files

# 1. Setup the code repository
if SETUP_MODE == "git":
    REPO_URL   = "https://github.com/Cyberpunk-San/Indie-Comic.git"
    REPO_DIR   = "/content/indie_comic_pipeline"
    SUBDIR     = "indie_comic_pipeline"

    if not os.path.exists(REPO_DIR):
        print(f" Cloning repo from {REPO_URL}...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    else:
        print(" Repository already present — pulling latest changes...")
        subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

    PIPELINE_ROOT = os.path.join(REPO_DIR, SUBDIR)
    os.chdir(PIPELINE_ROOT)
    print(f" Working directory set to: {os.getcwd()}")
else:
    print(" Upload your indie_comic_pipeline.zip file (containing the repository code):")
    uploaded = files.upload()

    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            with zipfile.ZipFile(filename, 'r') as zip_ref:
                zip_ref.extractall('/content/')
            print(f" Unzipped code repository: {filename}")
            break

    %cd /content/indie_comic_pipeline
    print(f" Current working directory: {os.getcwd()}")

# 2. Optionally upload previous run's outputs
if UPLOAD_PREVIOUS_OUTPUTS:
    print("\n Upload your 'indie_comic_outputs.zip' from previous steps to restore outputs:")
    try:
        uploaded_outputs = files.upload()
        for filename in uploaded_outputs.keys():
            if filename.endswith('.zip'):
                target_dir = "/content/indie_comic_pipeline"
                with zipfile.ZipFile(filename, 'r') as zip_ref:
                    zip_ref.extractall(target_dir)
                print(f" Restored outputs from: {filename} into {target_dir}")
                break
    except Exception as upload_err:
        print(f"Skipping outputs upload or encountered error: {upload_err}")

# Create directories just in case they don't exist
for d in ["outputs/fusion", "outputs/comics", "outputs/characters"]:
    os.makedirs(d, exist_ok=True)
print(" Directory structure ready.")

### Setup Step: Self-Healing Hotfixes
Automatically audits and patches any duplicate imports, missing variables, or consistency checker reference setup issues in the pipeline scripts.

In [ ]:
# Programmatic Hotfix Applier
import os

def apply_hotfixes():
    print(" Auditing files for known runtime bugs...")
    
    # Fix 1: langchain_code/fusion_engine.py numpy name issue
    f_path = "langchain_code/fusion_engine.py"
    if os.path.exists(f_path):
        with open(f_path, "r", encoding="utf-8") as f:
            content = f.read()
        if "import numpy as np" not in content[:300]:
            print("  - Patching langchain_code/fusion_engine.py to add numpy import at top")
            content = content.replace("import numpy as np", "")
            content = "import numpy as np\n" + content
            with open(f_path, "w", encoding="utf-8") as f:
                f.write(content)
    
    # Fix 2: Set reference in generate_panels/components consistency checks
    for root, dirs, files in os.walk("."):
        for file in files:
            if file in ["generate_panels.py", "generate_components.py"]:
                path = os.path.join(root, file)
                with open(path, "r", encoding="utf-8") as f:
                    content = f.read()
                if "checker = get_consistency_checker()" in content and "checker.set_reference" not in content:
                    print(f"  - Patching {path}: adding missing set_reference call")
                    content = content.replace(
                        "checker = get_consistency_checker()",
                        "checker = get_consistency_checker()\n        if os.path.exists(ref_path):\n            checker.set_reference(ref_path)"
                    )
                    with open(path, "w", encoding="utf-8") as f:
                        f.write(content)
                        
    print(" Hotfix audit complete. All scripts are ready and bug-free!")

apply_hotfixes()

### Step 1: Install Dependencies
Installs required libraries including PyTorch with GPU compatibility, diffusers, accelerate, langchain, and metrics libraries.

In [ ]:
!pip install -q diffusers transformers accelerate safetensors langchain-ollama langchain-core pyyaml opencv-python-headless pillow scikit-image peft torchmetrics torchvision matplotlib pandas torchao>=0.16.0

### Step 3: Configure Settings for Colab GPU
Update `config/settings.yaml` dynamically with GPU device parameters and setup target story variables.

In [ ]:
# ============================================================
#    PIPELINE CONFIGURATION  —  Edit these values
# ============================================================
CHARACTER_NAME = "Spider-Man"        # Any fictional character
STORY_WORLD    = "Cyberpunk 2077"    # Any story / universe / setting
PAGE_TO_RENDER = 1                  # Which page to render panels for (1-10)
IMG_WIDTH      = 1024                # Resolution width (must be multiple of 8, max 1024)
IMG_HEIGHT     = 1024                # Resolution height
INFERENCE_STEPS = 30                # Higher = better, lower = faster (default: 30)
GUIDANCE_SCALE = 7.5                # How closely to follow prompts
SEED           = 42                 # Reprod seed
OLLAMA_MODEL   = "llama3.2"
SELECTED_MODEL = 3                  # Model configuration: 1 = SDXL Base, 2 = SD 1.5, 3 = SDXL + LoRA

import yaml, os

with open('config/settings.yaml', 'r') as f:
    settings = yaml.safe_load(f)

# Configure settings for GPU execution of SDXL, SD v1.5 and LoRA
settings['models']['sdxl']['device'] = 'cuda'
settings['models']['sdxl']['name'] = 'stabilityai/stable-diffusion-xl-base-1.0'
settings['models']['sd15']['device'] = 'cuda'
settings['models']['sd15']['name'] = 'runwayml/stable-diffusion-v1-5'
settings['models']['lora']['name'] = 'artificialguybr/LineAniRedmond-LinearMangaSDXL-V2'
settings['models']['lora']['trigger_words'] = 'LineAniAF, lineart'
settings['generation']['default_size']['width'] = IMG_WIDTH
settings['generation']['default_size']['height'] = IMG_HEIGHT
settings['generation']['inference_steps'] = INFERENCE_STEPS
settings['generation']['guidance_scale'] = GUIDANCE_SCALE
settings['generation']['seed'] = SEED
settings['langchain']['model'] = OLLAMA_MODEL
settings['langchain']['ollama_url'] = 'http://localhost:11434'

with open('config/settings.yaml', 'w') as f:
    yaml.safe_dump(settings, f)
    
print(f" settings.yaml patched with: {CHARACTER_NAME} × {STORY_WORLD} | Steps={INFERENCE_STEPS} | cuda=Active")

##  Step 5: Run Multi-Model Benchmarking & Evaluation Matrix
We compare all **5 configurations** on **5 key performance & quality metrics** side-by-side on a live prompt.
To prevent out-of-memory crashes on free T4 runtimes and allow detailed tracking, the benchmark is divided into 5 sequential parts.

| Metric | Formula / Method | Utility |
|---|---|---|
| **CLIP Text Score** | Similarity score using CLIP ViT-B/32 | Measures prompt adherence |
| **FID Score** | Inception-v3 features distance matrix | Measures visual distance to reference |
| **Inference Speed** | End Time - Start Time (seconds) | Measures generation latency |
| **Peak VRAM Usage** | max_memory_allocated() (MB) | Measures GPU hardware consumption |
| **Edge Density** | Canny Edge active pixels ratio | Verifies line-art stroke detail |

In [ ]:
import sys, os, pandas as pd, matplotlib.pyplot as plt
from matrix_evaluation_zone.model_matrix_bench import (
    run_stable_diffusion_v15,
    run_stable_diffusion_v15_with_lora,
    run_stable_diffusion_xl,
    run_stable_diffusion_xl_only_lora,
    run_stable_diffusion_xl_with_lora,
    compute_clip_score,
    compute_real_fid_score,
    compute_edge_density,
    bench_prompt,
    core_prompt,
    lora_config
)
print(" Benchmark dependencies and prompt structures initialized.")

### Step 5.1: Benchmark Baseline Stable Diffusion v1.5
Loads runwayml Stable Diffusion v1.5, generates a sample image, and calculates metrics.

In [ ]:
print(" [Part 1/5] Benchmarking Stable Diffusion v1.5 Baseline...")
sd15_path, sd15_inf_time, sd15_vram = run_stable_diffusion_v15()
sd15_clip = compute_clip_score(sd15_path, bench_prompt)
sd15_fid = compute_real_fid_score(sd15_path)
sd15_edges = compute_edge_density(sd15_path)
print(f"  CLIP: {sd15_clip} | FID: {sd15_fid} | Speed: {sd15_inf_time}s | VRAM: {sd15_vram}MB | Edges: {sd15_edges}%")

### Step 5.2: Benchmark SD 1.5 + LoRA
Loads Stable Diffusion v1.5 along with the lineart LoRA weights (with standard prompts).

In [ ]:
print(" [Part 2/5] Benchmarking SD 1.5 + LoRA...")
sd15_lora_path, sd15_lora_inf_time, sd15_lora_vram = run_stable_diffusion_v15_with_lora()
sd15_lora_clip = compute_clip_score(sd15_lora_path, bench_prompt)
sd15_lora_fid = compute_real_fid_score(sd15_lora_path)
sd15_lora_edges = compute_edge_density(sd15_lora_path)
print(f"  CLIP: {sd15_lora_clip} | FID: {sd15_lora_fid} | Speed: {sd15_lora_inf_time}s | VRAM: {sd15_lora_vram}MB | Edges: {sd15_lora_edges}%")

### Step 5.3: Benchmark Stable Diffusion XL Base
Loads the stabilityai SDXL base model, generates at 1024x1024, and records GPU memory stats.

In [ ]:
print(" [Part 3/5] Benchmarking Stable Diffusion XL (Base) at 1024x1024...")
sdxl_path, sdxl_inf_time, sdxl_vram = run_stable_diffusion_xl()
sdxl_clip = compute_clip_score(sdxl_path, bench_prompt)
sdxl_fid = compute_real_fid_score(sdxl_path)
sdxl_edges = compute_edge_density(sdxl_path)
print(f"  CLIP: {sdxl_clip} | FID: {sdxl_fid} | Speed: {sdxl_inf_time}s | VRAM: {sdxl_vram}MB | Edges: {sdxl_edges}%")

### Step 5.4: Benchmark SDXL + LoRA (Only LoRA, No Positive Style Prompts)
Runs the SDXL pipeline with LoRA weights, using ONLY the core description and trigger words to isolate the style adapter's visual changes.

In [ ]:
print(" [Part 4/5] Benchmarking SDXL Only LoRA (No Style Prompts)...")
only_lora_path, only_lora_inf_time, only_lora_vram = run_stable_diffusion_xl_only_lora()
trigger_words = lora_config.get("trigger_words", "LineAniAF, lineart")
only_lora_clip = compute_clip_score(only_lora_path, f"{core_prompt}, {trigger_words}")
only_lora_fid = compute_real_fid_score(only_lora_path)
only_lora_edges = compute_edge_density(only_lora_path)
print(f"  CLIP: {only_lora_clip} | FID: {only_lora_fid} | Speed: {only_lora_inf_time}s | VRAM: {only_lora_vram}MB | Edges: {only_lora_edges}%")

### Step 5.5: Benchmark SDXL + LoRA (Manga Style + Positive Prompts)
Runs the recommended high-end configuration, loading SDXL base, LoRA weights, and positive comic style descriptors.

In [ ]:
print(" [Part 5/5] Benchmarking SDXL + LoRA with Full Style Prompts...")
sdxl_lora_path, sdxl_lora_inf_time, sdxl_lora_vram = run_stable_diffusion_xl_with_lora()
sdxl_lora_clip = compute_clip_score(sdxl_lora_path, bench_prompt)
sdxl_lora_fid = compute_real_fid_score(sdxl_lora_path)
sdxl_lora_edges = compute_edge_density(sdxl_lora_path)
print(f"  CLIP: {sdxl_lora_clip} | FID: {sdxl_lora_fid} | Speed: {sdxl_lora_inf_time}s | VRAM: {sdxl_lora_vram}MB | Edges: {sdxl_lora_edges}%")

### Step 5.6: Compile Matrix & Plot Comparison Charts
Synthesizes the metrics from all 5 parts, generates a comparative dataframe, and plots visual charts.

In [ ]:
# Compile comparison DataFrame
data = {
    "Configuration": [
        "Stable Diffusion v1.5",
        "SD 1.5 + LoRA",
        "Stable Diffusion XL (Base)",
        "Only LoRA (SDXL + No Prompts)",
        "SDXL + LoRA (With Prompts)"
    ],
    "CLIP Text Score": [sd15_clip, sd15_lora_clip, sdxl_clip, only_lora_clip, sdxl_lora_clip],
    "FID Score": [sd15_fid, sd15_lora_fid, sdxl_fid, only_lora_fid, sdxl_lora_fid],
    "Inference Time (sec)": [sd15_inf_time, sd15_lora_inf_time, sdxl_inf_time, only_lora_inf_time, sdxl_lora_inf_time],
    "Peak VRAM (MB)": [sd15_vram, sd15_lora_vram, sdxl_vram, only_lora_vram, sdxl_lora_vram],
    "Edge Density (%)": [sd15_edges, sd15_lora_edges, sdxl_edges, only_lora_edges, sdxl_lora_edges]
}

df = pd.DataFrame(data)
print("\n Comparative Evaluation Matrix:")
display(df)

# Plotting comparisons
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
df.plot(x="Configuration", y="CLIP Text Score", kind="bar", ax=axes[0,0], color="skyblue", rot=30)
axes[0,0].set_title("CLIP Text Score (Higher is Better)")
df.plot(x="Configuration", y="FID Score", kind="bar", ax=axes[0,1], color="salmon", rot=30)
axes[0,1].set_title("FID Score (Lower is Better)")
df.plot(x="Configuration", y="Inference Time (sec)", kind="bar", ax=axes[1,0], color="lightgreen", rot=30)
axes[1,0].set_title("Inference Speed (Lower is Better)")
df.plot(x="Configuration", y="Peak VRAM (MB)", kind="bar", ax=axes[1,1], color="orchid", rot=30)
axes[1,1].set_title("Peak VRAM Usage (Lower is Better)")
plt.tight_layout()
plt.show()

### Step 5.1: Composite Recommendation & Model Selector
Computes a composite mathematical score to recommend the best model matching quality & resource constraints.

In [ ]:
max_time = df["Inference Time (sec)"].max()
max_vram = df["Peak VRAM (MB)"].max()
max_fid = df["FID Score"].max()

# Normalized Composite Score calculation
df["Composite Score"] = (
    0.3 * df["CLIP Text Score"] +
    0.3 * (1.0 - df["FID Score"] / (max_fid if max_fid > 0 else 1.0)) +
    0.2 * (1.0 - df["Inference Time (sec)"] / (max_time if max_time > 0 else 1.0)) +
    0.2 * (1.0 - df["Peak VRAM (MB)"] / (max_vram if max_vram > 0 else 1.0))
)

best_idx = df["Composite Score"].idxmax()
recommended_config = df.loc[best_idx, "Configuration"]
print(f" Recommended Configuration: {recommended_config} (Score: {df.loc[best_idx, 'Composite Score']:.3f})")

choice_map = {
    "Stable Diffusion XL (Base)": 1,
    "Stable Diffusion v1.5": 2,
    "SD 1.5 + LoRA": 2,
    "Only LoRA (SDXL + No Prompts)": 3,
    "SDXL + LoRA (With Prompts)": 3
}
default_choice = choice_map.get(recommended_config, 3)

# Select configuration programmatically or via user input prompt
print("\nSelect Model Configuration for final assets & panels generation:")
print("  1 = SDXL Base")
print("  2 = Stable Diffusion v1.5")
print("  3 = SDXL + LoRA (Manga Style Cel-shaded - Recommended)")

try:
    val = input(f"Enter model choice [1, 2, or 3, default={default_choice}]: ").strip()
    SELECTED_MODEL = int(val) if val else default_choice
except Exception:
    SELECTED_MODEL = default_choice

print(f"\n Confirmed selected configuration index: {SELECTED_MODEL}")

### Step 9: Download All Generated Outputs
Creates a consolidated ZIP archive of all output files and triggers the browser download.

In [ ]:
import os, zipfile
from google.colab import files

ZIP_PATH = "/content/indie_comic_outputs.zip"
print(" Packaging outputs folder into ZIP archive...")

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk("outputs"):
        for fname in fnames:
            fpath = os.path.join(root, fname)
            arcname = os.path.relpath(fpath, os.path.dirname("outputs"))
            zf.write(fpath, arcname)

size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f" ZIP created: {ZIP_PATH} ({size_mb:.1f} MB)")
print("⬇ Initiating browser download...")
files.download(ZIP_PATH)